# Common topics
- The Aim of this notebook is to create topic clusters starting from different models to determine the common topics.
- The similarity metric used in this notebook is topic closeness.
- See the original paper at https://arxiv.org/abs/2412.18376

## Loading libraries

In [2]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
import networkx as nx
import leidenalg
import igraph as ig
from collections import defaultdict

## Loading models

In [4]:
MAGAZINE_1 = 'scopus'
DATASET_TEXT_FEATURE = (
    "text"  # In the dataset file, the column name that contains the text data
)
cfg_dict_1 = cfg.MAGAZINE_CONFIG[MAGAZINE_1]

In [9]:
MAGAZINE_2 = 'the_guardian'
cfg_dict_2 = cfg.MAGAZINE_CONFIG[MAGAZINE_2]

In [10]:
MAGAZINE_3 = 'science_news'
cfg_dict_3 = cfg.MAGAZINE_CONFIG[MAGAZINE_3]

In [ ]:
from bertopic import BERTopic

model_path = cfg_dict_1['REFERENCE_MODEL']
model_1 = BERTopic.load(model_path, 
                      embedding_model=cfg.EMBEDDING_MODEL
                      )

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_path_2 = cfg_dict_2['REFERENCE_MODEL']

model_2 = BERTopic.load(model_path_2,
                        cfg.EMBEDDING_MODEL
                        )


In [ ]:
model_path_3 = cfg_dict_3['REFERENCE_MODEL']

model_3 = BERTopic.load(model_path_3,
                        cfg.EMBEDDING_MODEL
                        )


## Loading data

In [15]:
data = np.load(cfg_dict_1['OUTPUT_PATH'],allow_pickle=True) 

ids = data['id']
texts = data['text'] 
embeddings = data['embedding'] 
documents = data['clean_text']

In [ ]:
data_2 = np.load(cfg_dict_2['OUTPUT_PATH'],allow_pickle=True) 

ids_2 = data_2['id']
texts_2 = data_2['text'] 
embeddings_2 = data_2['embedding'] 
documents_2 = data_2['clean_text']

In [17]:
data_3 = np.load(cfg_dict_3['OUTPUT_PATH'],allow_pickle=True) 

ids_3 = data_3['id']
texts_3 = data_3['text'] 
embeddings_3 = data_3['embedding'] 
documents_3 = data_3['clean_text']

## BTM Metrics evaluation - Model 1 vs Model 2

### First side evalutation - Model 1 -> Model 2

In [ ]:
from pipeline.src.python.btm import BTM

models_metrics_1_vs_2 = BTM(model_1=model_1,
                            model_2=model_2,
                            ids_1=ids,
                            texts_1=texts,
                            embeddings_1=embeddings,
                            texts_2=texts_2,
                            embeddings_2=embeddings_2,
                            model_1_name=MAGAZINE_1.title(),
                            model_2_name=MAGAZINE_2.title())

2026-02-09 14:09:37,164 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:37,370 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3820500373840332


In [25]:
models_metrics_1_vs_2.evaluate_metrics()

In [26]:
model_1_vs_model_2_closeness = models_metrics_1_vs_2.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Second side evaluation - Model 2 -> Model 1

In [ ]:
inverse_models_metrics_2_vs_1 = BTM(model_1=model_2,
                                    model_2=model_1,
                                    ids_1=ids_2,
                                    texts_1=texts_2,
                                    embeddings_1=embeddings_2,
                                    texts_2=texts,
                                    embeddings_2=embeddings,
                                    model_1_name=MAGAZINE_2.title(),model_2_name=MAGAZINE_1.title())

2026-02-09 14:09:40,871 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:42,092 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.38246655464172363


In [28]:
inverse_models_metrics_2_vs_1.evaluate_metrics()

In [29]:
model_2_vs_model_1_closeness = inverse_models_metrics_2_vs_1.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

## BTM Metrics evaluation - Model 1 vs Model 3

### First side evaluation - Model 1 -> Model 3

In [ ]:
from pipeline.src.python.btm import BTM

models_metrics_1_vs_3 = BTM(model_1=model_1,
                            model_2=model_3,
                            ids_1=ids,
                            texts_1=texts,
                            embeddings_1=embeddings,
                            texts_2=texts_3,
                            embeddings_2=embeddings_3,
                            model_1_name=MAGAZINE_1.title(),
                            model_2_name=MAGAZINE_3.title())

2026-02-09 14:09:54,709 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:54,721 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3790895342826843


In [31]:
models_metrics_1_vs_3.evaluate_metrics()

In [32]:
model_1_vs_model_3_closeness = models_metrics_1_vs_3.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Second side evaluation - Model 3 -> Model 1

In [ ]:
inverse_models_metrics_3_vs_1 = BTM(model_1=model_3,
                                    model_2=model_1,
                                    ids_1=ids_3,
                                    texts_1=texts_3,
                                    embeddings_1=embeddings_3,
                                    texts_2=texts,
                                    embeddings_2=embeddings,
                                    model_1_name=MAGAZINE_3.title(),
                                    model_2_name=MAGAZINE_1.title())

2026-02-09 14:11:39,363 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:11:40,577 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.38246655464172363


In [34]:
inverse_models_metrics_3_vs_1.evaluate_metrics()

In [35]:
model_3_vs_model_1_closeness = inverse_models_metrics_3_vs_1.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

## BTM Metrics evaluation - Model 2 vs Model 3

### First side evaluation - Model 2 -> Model 3

In [36]:
from pipeline.src.python.btm import BTM

models_metrics_2_vs_3 = BTM(model_1=model_2,
                            model_2=model_3,
                            ids_1=ids_2,
                            texts_1=texts_2,
                            embeddings_1=embeddings_2,
                            texts_2=texts_3,
                            embeddings_2=embeddings_3,
                            model_1_name=MAGAZINE_2.title(),
                            model_2_name=MAGAZINE_3.title()
                            )

2026-02-09 14:14:37,245 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:14:37,257 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3790895342826843


In [37]:
models_metrics_2_vs_3.evaluate_metrics()

In [38]:
model_2_vs_model_3_closeness = models_metrics_2_vs_3.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Second side evaluation - Model 3 -> Model 2

In [39]:
inverse_models_metrics_3_vs_2 = BTM(model_1=model_3,
                                    model_2=model_2,
                                    ids_1=ids_3,
                                    texts_1=texts_3,
                                    embeddings_1=embeddings_3,
                                    texts_2=texts_2,
                                    embeddings_2=embeddings_2,
                                    model_1_name=MAGAZINE_3.title(),
                                    model_2_name=MAGAZINE_2.title())

2026-02-09 14:16:10,197 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:16:10,400 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3820500373840332


In [41]:
inverse_models_metrics_3_vs_2.evaluate_metrics()

In [40]:
model_3_vs_model_2_closeness = inverse_models_metrics_3_vs_2.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

## Saving data to parquet

### Standardize column names

In [ ]:
model_1_vs_model_2_closeness = model_1_vs_model_2_closeness.rename(columns={f'{MAGAZINE_1.title()} Topic Label':'Model 1 Label',
                                                                            f'{MAGAZINE_2.title()} Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_2_vs_model_1_closeness = model_2_vs_model_1_closeness.rename(columns={f'{MAGAZINE_1.title()} Topic Label':'Model 2 Label',
                                                                            f'{MAGAZINE_2.title()} Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_1_vs_model_3_closeness = model_1_vs_model_3_closeness.rename(columns={f'{MAGAZINE_1.title()} Topic Label':'Model 1 Label',
                                                                            f'{MAGAZINE_3.title()} Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_3_vs_model_1_closeness = model_3_vs_model_1_closeness.rename(columns={f'{MAGAZINE_1.title()} Topic Label':'Model 2 Label',
                                                                            f'{MAGAZINE_3.title()} Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_2_vs_model_3_closeness = model_2_vs_model_3_closeness.rename(columns={f'{MAGAZINE_2.title()} Topic Label':'Model 1 Label',
                                                                            f'{MAGAZINE_3.title()} Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_3_vs_model_2_closeness = model_3_vs_model_2_closeness.rename(columns={f'{MAGAZINE_2.title()} Topic Label':'Model 2 Label',
                                                                            f'{MAGAZINE_3.title()} Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

### Create and save edges dataset

In [58]:
closeness = pd.concat([model_1_vs_model_2_closeness,
                model_2_vs_model_1_closeness,
                model_1_vs_model_3_closeness,
                model_3_vs_model_1_closeness,
                model_2_vs_model_3_closeness,
                model_3_vs_model_2_closeness])


In [63]:
closeness.to_parquet('edges.parquet')

### Create and save nodes dataset

In [ ]:
counts = model_1.get_topic_freq()['Count'][1:].to_list() if -1 in model_1.topics_ else model_1.get_topic_freq()['Count'].to_list()
labels  = model_1.custom_labels_[1:] if -1 in model_1.topics_ else model_1.custom_labels_

model_1_nodes = pd.DataFrame({'Topic Label': labels,
                              'Degree':counts })

model_1_nodes['Model'] = MAGAZINE_1.title()


In [28]:
counts_2 = model_2.get_topic_freq()['Count'][1:].to_list() if -1 in model_2.topics_ else model_2.get_topic_freq()['Count'].to_list()
labels_2  = model_2.custom_labels_[1:] if -1 in model_2.topics_ else model_2.custom_labels_

model_2_nodes = pd.DataFrame({'Topic Label': labels_2,
                              'Degree':counts_2 })
model_2_nodes['Model'] = MAGAZINE_2.title()

In [30]:
counts_3 = model_3.get_topic_freq()['Count'][1:].to_list() if -1 in model_3.topics_ else model_3.get_topic_freq()['Count'].to_list()
labels_3  = model_3.custom_labels_[1:] if -1 in model_3.topics_ else model_3.custom_labels_

model_3_nodes = pd.DataFrame({'Topic Label': labels_3,
                              'Degree':counts_3 })
model_3_nodes['Model'] = MAGAZINE_3.title()

In [32]:
nodes = pd.concat([model_1_nodes,
                model_2_nodes,
                model_3_nodes])

In [33]:
nodes.to_parquet('nodes.parquet')

## Clustering Analysis

### Retriving data from parquet

In [10]:
import pandas as pd
edges = pd.read_parquet('edges.parquet')
nodes = pd.read_parquet('nodes.parquet')

### Loop to decide the PRIMARY_COMMUNITY_THRESHOLD

In [5]:
community_list = []
START = 0.05
EPS = 1e-9
END = 0.4 + EPS
STEP = 0.0125
threshold_list = np.arange(START,END,STEP)


In [6]:
enriched_edges = edges.merge(nodes,how='inner',left_on='Model 1 Label',right_on='Topic Label')
enriched_edges['N_Match'] = enriched_edges['Topic Closeness']  * enriched_edges['Degree']
enriched_edges = enriched_edges[ enriched_edges['N_Match'] > 5 ]

In [7]:
for i in threshold_list:
    filtered_topics_1 = []
    filtered_topics_2 = []
    closeness = []

    for _, edge in enriched_edges.iterrows():

        topic1 = edge['Model 1 Label']
        topic2 = edge['Model 2 Label']
        c = edge['Topic Closeness']

        if c <= i:
            continue

        bidirection = enriched_edges[
            (enriched_edges['Model 1 Label'] == topic2) &
            (enriched_edges['Model 2 Label'] == topic1) &
            (enriched_edges['Topic Closeness'] > i)
        ]

        if not bidirection.empty:

            filtered_topics_1.append(topic1)
            filtered_topics_2.append(topic2)
            closeness.append(c)

        filtered_edges = pd.DataFrame({'Model 1 Label': filtered_topics_1,
                        'Model 2 Label': filtered_topics_2,
                        'Topic Closeness':closeness})
        
    G = nx.DiGraph(description='Connected components') 

    for _, node in nodes.iterrows():
        G.add_node(
        node['Topic Label'],
        degree=node['Degree'],
        model=node['Model']
    )

    for _, edge in filtered_edges.iterrows():
        G.add_edge(
        edge['Model 1 Label'],
        edge['Model 2 Label'],
        weight=edge['Topic Closeness']
    )
        
    dir_weights = {}

    for u, v, data in G.edges(data=True):
        w = data["weight"]
        dir_weights[(u, v)] = w

    G_und = nx.Graph()

    for (u, v), w_uv in dir_weights.items():

        if (v, u) in dir_weights:
            w_vu = dir_weights[(v, u)]

            w = min(w_uv, w_vu)
            #w = harmonic_mean(w_uv, w_vu)

            if not G_und.has_edge(u, v):
                G_und.add_edge(u, v, weight=w)
                
    nodes_list = list(G_und.nodes())
    G_ig = ig.Graph.from_networkx(G_und)
    G_ig.vs["name"] = nodes_list

    partition = leidenalg.find_partition(
    G_ig,
    leidenalg.ModularityVertexPartition,
    weights="weight",
    n_iterations=-1
    )

    clusters = partition.membership

    node_to_cluster = {
    v["name"]: clusters[i]
    for i, v in enumerate(G_ig.vs)
    }

    clusters = defaultdict(list)

    for node, comm in node_to_cluster.items():
        clusters[comm].append(node)

    label_name = []
    cluster_number = []
    for k,v in node_to_cluster.items():
        label_name.append(k)
        cluster_number.append(v)
    
    community_list.append(sorted(cluster_number)[-1]+1)


In [8]:
threshold_list[ np.argmax(community_list) ]

np.float64(0.11249999999999999)

### Edge filtering

- Filter all the edges that don't have bidirectional linkage
- Filter all the edges that have a Topic Closeness less than 0.1
- Filter all the edges with less than 5 match

In [11]:
enriched_edges = edges.merge(nodes,how='inner',left_on='Model 1 Label',right_on='Topic Label')

In [12]:
enriched_edges['N_Match'] = enriched_edges['Topic Closeness']  * enriched_edges['Degree']

In [13]:
enriched_edges = enriched_edges[ enriched_edges['N_Match'] > 5 ]

In [14]:
PRIMARY_COMMUNITY_THRESHOLD = 0.1

In [15]:
filtered_topics_1 = []
filtered_topics_2 = []
closeness = []

for _, edge in enriched_edges.iterrows():

    topic1 = edge['Model 1 Label']
    topic2 = edge['Model 2 Label']
    c = edge['Topic Closeness']

    if c <= PRIMARY_COMMUNITY_THRESHOLD:
        continue

    bidirection = enriched_edges[
        (enriched_edges['Model 1 Label'] == topic2) &
        (enriched_edges['Model 2 Label'] == topic1) &
        (enriched_edges['Topic Closeness'] > PRIMARY_COMMUNITY_THRESHOLD)
    ]

    if not bidirection.empty:

        filtered_topics_1.append(topic1)
        filtered_topics_2.append(topic2)
        closeness.append(c)

In [16]:
filtered_edges = pd.DataFrame({'Model 1 Label': filtered_topics_1,
                        'Model 2 Label': filtered_topics_2,
                        'Topic Closeness':closeness})

### Graph creation

In [17]:
G = nx.DiGraph(description='Connected components') 

for _, node in nodes.iterrows():
    G.add_node(
    node['Topic Label'],
    degree=node['Degree'],
    model=node['Model']
)

for _, edge in filtered_edges.iterrows():
    G.add_edge(
    edge['Model 1 Label'],
    edge['Model 2 Label'],
    weight=edge['Topic Closeness']
)

### Similarity operation

We need to transform directed graph into undirected graph.
To do it, we need to decide how combine the weights of the two edges.

The options are:
- Min
- Harmonic mean

To be more conservative, we adopt the min between two weights.

In [18]:
dir_weights = {}

for u, v, data in G.edges(data=True):
    w = data["weight"]
    dir_weights[(u, v)] = w

In [19]:
def harmonic_mean(a, b, eps=1e-9):
    return 2*a*b/(a+b+eps)

In [20]:
G_und = nx.Graph()

for (u, v), w_uv in dir_weights.items():

    if (v, u) in dir_weights:
        w_vu = dir_weights[(v, u)]

        w = min(w_uv, w_vu)
        #w = harmonic_mean(w_uv, w_vu)

        if not G_und.has_edge(u, v):
            G_und.add_edge(u, v, weight=w)


### Community algorithm execution

We adopt the Leiden algorithm to build a primary community structure.

See more about it on https://medium.com/@swapnil.agashe456/leiden-clustering-for-community-detection-a-step-by-step-guide-with-python-implementation-c883933a1430

In [21]:
nodes = list(G_und.nodes())
G_ig = ig.Graph.from_networkx(G_und)
G_ig.vs["name"] = nodes

In [22]:
partition = leidenalg.find_partition(
    G_ig,
    leidenalg.ModularityVertexPartition,
    weights="weight",
    n_iterations=-1
)

In [23]:
clusters = partition.membership

In [24]:
node_to_cluster = {
    v["name"]: clusters[i]
    for i, v in enumerate(G_ig.vs)
}

In [25]:
clusters = defaultdict(list)

for node, comm in node_to_cluster.items():
    clusters[comm].append(node)

for c, nodes in clusters.items():
    print(f"Cluster {c}:")
    print(", ".join(nodes))
    print()


Cluster 24:
Tuberculosis Resistance and Control, Tuberculosis Treatment Challenges

Cluster 0:
Zoonotic Disease Surveillance, Disease Spread and Environmental Impact, WNV Surveillance and Detection, Zika Outbreak and Mosquito Spread, Zika Virus Outbreaks, Zika Virus Outbreak Risk, Zika Virus and Microcephaly, Climate and Health Impact, Disease Outbreak Tracking Initiative

Cluster 8:
Infection Control and Prevention, Hospital Infection Control Improvement, Hospital Surface Sterilization and Contamination Control, MRSA Hospital Surveillance

Cluster 1:
SARS-CoV-2 Genomic Evolution, Coronavirus Death Trends in England, Covid-19 Pandemic Mitigation Strategies, COVID-19 Spread and Mortality, Global Coronavirus Outbreak Death Toll, School Outbreak Transmission, School Reopening and Educational Needs, Coronavirus Death Toll Surpasses Million

Cluster 4:
HPAI Outbreak and Pathogenicity, H5N1 Bird Flu Outbreak in Poultry Industry, Flu Virus Spread Patterns, Viral Replication in Influenza Virus

In [26]:
label_name = []
cluster_number = []
for k,v in node_to_cluster.items():
    label_name.append(k)
    cluster_number.append(v)


In [27]:
print(f"Number of communities: {sorted(cluster_number)[-1]+1}")

Number of communities: 45


In [28]:
leiden_cluster = pd.DataFrame({'Topic Label':label_name,
              'Cluster': cluster_number})

In [29]:
leiden_cluster['Insertion_run'] = 0

### Enrichment phase

Now, we enrich the primary community structure using the similarity between excluded topics and cluster members.

In [30]:
# The threshold that permit to decide when stop the insertion of unclustered topics
MEAN_CLOSENESS_THRESHOLD = 0.1
TOPICS_TO_ADD_EACH_RUN = 1

leiden_cluster.sort_values(by='Cluster').to_csv(f'primary_community_structure.csv',sep='ç',encoding='utf-8')
print('Primary community structure saved')

Primary community structure saved


In [31]:
## Enrichment loop
i = 1

while True:
    # Add the cluster size to the dataframe
    leiden_cluster = leiden_cluster.merge(leiden_cluster['Cluster'].value_counts().reset_index().rename(columns={'count':'Cluster size'}),on='Cluster')

    # Consider only topics without cluster
    unclustered_topics = edges[ ~ edges['Model 1 Label'].isin(leiden_cluster['Topic Label'])]

    # Consider only topics with at least one edge with clustered topics
    unclustered_topics_with_closeness = unclustered_topics.merge(leiden_cluster,how='left',left_on='Model 2 Label',right_on='Topic Label')
    unclustered_topics_with_closeness = unclustered_topics_with_closeness[ ~ unclustered_topics_with_closeness['Topic Label'].isna()]

    # For each unclustered topic, sum the best 3 edges with the cluster members
    k = 3
    top_k_closeness = ( 
    unclustered_topics_with_closeness
    .groupby(['Model 1 Label','Cluster','Cluster size'],as_index=False)
    .head(k) 
    .groupby(['Model 1 Label','Cluster','Cluster size'])['Topic Closeness']
    .agg(Topic_Closeness_Sum="sum")
    )

    # Compute the mean by k (even if there aren't enough connections)
    top_k_closeness = top_k_closeness.reset_index()
    
    top_k_closeness['Topic Closeness Mean'] = top_k_closeness['Topic_Closeness_Sum'] / k

    unclustered_topics_with_closeness_aggregated = top_k_closeness.copy()

    # Aggregate and select the best cluster that has the best mean score
    the_best_unclustered_topics = unclustered_topics_with_closeness_aggregated.loc[ 
    unclustered_topics_with_closeness_aggregated.groupby(['Model 1 Label'])["Topic Closeness Mean"].idxmax()
    ]
    
    the_best_unclustered_topics = the_best_unclustered_topics.astype({'Cluster':"int64","Cluster size":"int64"})

    # Select the best 10 uncluster topic and add to appropriate clusters
    added_topics = the_best_unclustered_topics[ the_best_unclustered_topics['Topic Closeness Mean'] >= MEAN_CLOSENESS_THRESHOLD ].sort_values(by='Topic Closeness Mean',ascending=False)[['Model 1 Label','Cluster','Cluster size']]
    added_topics = added_topics.rename(columns={'Model 1 Label':'Topic Label'})
    added_topics['Insertion_run'] = i
    leiden_cluster = pd.concat([leiden_cluster,added_topics.head(TOPICS_TO_ADD_EACH_RUN)])

    leiden_cluster = leiden_cluster.drop(columns=['Cluster size'])

    # Save the run outcome
    if len(added_topics) > 0:
        print(f'Run {i}: {len(added_topics.head(TOPICS_TO_ADD_EACH_RUN))} added')
        leiden_cluster.sort_values(by='Cluster').to_csv(f'community_enrichment_run_{i}.csv',sep='ç',encoding='utf-8')
        i += 1
    else:
        print(f'All the unclustered topics below the {MEAN_CLOSENESS_THRESHOLD} threshold were assigned to the clusters')
        break

Run 1: 1 added
Run 2: 1 added
Run 3: 1 added
Run 4: 1 added
Run 5: 1 added
Run 6: 1 added
Run 7: 1 added
Run 8: 1 added
Run 9: 1 added
Run 10: 1 added
Run 11: 1 added
Run 12: 1 added
Run 13: 1 added
Run 14: 1 added
Run 15: 1 added
Run 16: 1 added
Run 17: 1 added
Run 18: 1 added
Run 19: 1 added
Run 20: 1 added
Run 21: 1 added
Run 22: 1 added
Run 23: 1 added
Run 24: 1 added
Run 25: 1 added
Run 26: 1 added
Run 27: 1 added
Run 28: 1 added
Run 29: 1 added
Run 30: 1 added
Run 31: 1 added
Run 32: 1 added
Run 33: 1 added
Run 34: 1 added
Run 35: 1 added
Run 36: 1 added
Run 37: 1 added
Run 38: 1 added
Run 39: 1 added
Run 40: 1 added
Run 41: 1 added
Run 42: 1 added
Run 43: 1 added
Run 44: 1 added
Run 45: 1 added
Run 46: 1 added
Run 47: 1 added
Run 48: 1 added
Run 49: 1 added
Run 50: 1 added
Run 51: 1 added
Run 52: 1 added
Run 53: 1 added
Run 54: 1 added
Run 55: 1 added
Run 56: 1 added
Run 57: 1 added
Run 58: 1 added
Run 59: 1 added
Run 60: 1 added
Run 61: 1 added
Run 62: 1 added
Run 63: 1 added
R

### Cluster Label Assignment

In [34]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [35]:
model_name ="Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 5/5 [02:13<00:00, 26.74s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
def evaluate_topic_assignment(keywords_list):
    messages = [
        {
            "role": "user",
            "content": (
                "Create a short topic cluster label from the topic labels below.\n"
                "Return ONLY the label as a short noun phrase (3–6 words).\n"
                
                "Topic Label:\n"
                f"{'\n'.join(f'- {key}' for key in keywords_list)}\n\n"
            )
        }
    ]

    print(messages)


    testo = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([testo],
                              return_tensors="pt",
                              truncation=True,
                              max_length=2048).to(model.device)
    with torch.inference_mode():
        generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=24,
        do_sample = False,
        temperature = 0.0,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(generated_ids[0][len(model_inputs.input_ids[0]):],
                            skip_special_tokens=True)

    return text.strip().splitlines()[0].lstrip("-• ").strip()


In [26]:
n_clusters = leiden_cluster['Cluster'].nunique()

In [ ]:
label_dict = { i:None for i in range(n_clusters) }

for i in range(n_clusters):
    labels = leiden_cluster[ leiden_cluster['Cluster'] == i ]['Topic Label'].to_list()
    cluster_label=evaluate_topic_assignment(labels)
    label_dict[i] = cluster_label


[{'role': 'user', 'content': 'Create a short topic cluster label from the topic labels below.\nReturn ONLY the label as a short noun phrase (3–6 words).\nTopic Label:\n- Zoonotic Disease Surveillance\n- Disease Spread and Environmental Impact\n- WNV Surveillance and Detection\n- Zika Outbreak and Mosquito Spread\n- Zika Virus Outbreaks\n- Zika Virus Outbreak Risk\n- Zika Virus and Microcephaly\n- Climate and Health Impact\n- Disease Outbreak Tracking Initiative\n- Wastewater Surveillance for Cov-2 Detection\n- Dengue, Chikungunya, and Zika Arbovirus Surveillance\n- Global Biosurveillance Strategy\n- AI-Powered Pandemic Surveillance\n- Chikungunya Outbreak Transmission\n- Pandemic Impact and Hope\n- Travel Medicine and Infectious Disease Surveillance\n- Outbreak Detection Surveillance System\n- Wearable Health Monitoring System\n- Invasive Aedes Surveillance\n- Vanishing Half: Poetry and Narrative in Lockdown\n\n'}]
Cluster 0: Infectious Disease and Surveillance
[{'role': 'user', 'conte

In [43]:
leiden_cluster['Cluster Label'] = leiden_cluster['Cluster'].apply(lambda x: label_dict[x])

### Save the results in parquet file

In [ ]:
leiden_cluster['Model'] = leiden_cluster['Topic Label'].apply(lambda x: G.nodes[x]['model'].lower())

In [46]:
leiden_cluster.to_parquet('community_detection_results.parquet')

### Draw communities

In [32]:
edges = pd.read_parquet('edges.parquet')
nodes = pd.read_parquet('nodes.parquet')

In [33]:
G_display = nx.DiGraph(description='Connected components') 

for _, node in nodes.iterrows():
    G_display.add_node(
    node['Topic Label'],
    degree=node['Degree'],
    model=node['Model']
)

for _, edge in edges.iterrows():
    G_display.add_edge(
    edge['Model 1 Label'],
    edge['Model 2 Label'],
    weight=edge['Topic Closeness']
)

In [34]:
cluster_dict = leiden_cluster.set_index("Topic Label")['Cluster'].to_dict()

In [81]:
leiden_cluster.head(20)

,Topic Label,Cluster,Insertion_run
0,Tuberculosis Resistance and Control,24,0
1,Tuberculosis Treatment Challenges,24,0
2,Zoonotic Disease Surveillance,0,0
3,Disease Spread and Environmental Impact,0,0
4,Infection Control and Prevention,8,0
5,Hospital Infection Control Improvement,8,0
6,SARS-CoV-2 Genomic Evolution,1,0
7,Coronavirus Death Trends in England,1,0
8,Covid-19 Pandemic Mitigation Strategies,1,0
9,HPAI Outbreak and Pathogenicity,4,0


In [35]:
for node in G_display.nodes():
    G_display.nodes[node]['cluster'] = cluster_dict.get(node,-1)

In [36]:
insertion_dict = leiden_cluster.set_index("Topic Label")['Insertion_run'].to_dict()

In [37]:
for node in G_display.nodes():
    G_display.nodes[node]['position'] = insertion_dict.get(node,-1)

In [38]:
import matplotlib as mpl
cmap = mpl.pyplot.cm.tab20

for node in G_display.nodes():
    cluster = G_display.nodes[node].get("cluster",-1)
    if cluster == -1:
        G_display.nodes[node]['color'] = 'black'
    else:
        G_display.nodes[node]['color'] = mpl.colors.to_hex(cmap(cluster % 20))
    

In [77]:
import networkx as nx
from pyvis.network import Network


def scale(v, vmin, vmax, out_min, out_max):
    """Scala un valore da un range a un altro."""
    if vmax == vmin:
        return (out_min + out_max) / 2
    return out_min + (v - vmin) * (out_max - out_min) / (vmax - vmin)


def export_graph_to_html(
    G, 
    filename="graph.html",
    min_weight=0.0,
    hide_isolates=True,
    height="800px",
    width="100%",
    physics_enabled=True,
    show_buttons=True,
    filter_cluster=None,
    consider_outlier=False,
    size_threshold=-1,
):
    """
    Esporta un grafo NetworkX in un file HTML interattivo standalone.
    
    Parametri:
    ----------
    G : networkx.Graph
        Il grafo da esportare
    filename : str
        Nome del file HTML di output (default: "graph.html")
    min_weight : float
        Soglia minima per mostrare gli archi (default: 0.0)
    hide_isolates : bool
        Se True, nasconde i nodi senza archi dopo il filtro (default: True)
    height : str
        Altezza del grafo (default: "800px")
    width : str
        Larghezza del grafo (default: "100%")
    physics_enabled : bool
        Se True, abilita la simulazione fisica (default: True)
    show_buttons : bool
        Se True, mostra i controlli interattivi (default: True)
    
    Returns:
    --------
    dict : Statistiche del grafo esportato
    """

    icon_map = {"Scopus": "●", "Science_News": "▲", "The_Guardian": "✖"}

    if consider_outlier and filter_cluster:
        filter_cluster.append(-1)

    # 1) FILTRA GLI ARCHI in base al peso minimo e al cluster
    edges_kept = []
    for u, v, a in G.edges(data=True):
        # Cluster isolation
        if filter_cluster:
            if ( G.nodes[u].get('cluster',-1) not in filter_cluster ) | ( G.nodes[v].get('cluster',-1) not in filter_cluster ) | (  G.nodes[u].get('cluster',-1) == -1 & G.nodes[v].get('cluster',-1) == -1  ) :
                continue

        if size_threshold != -1:
            if ( G.nodes[u].get('degree',0) <= size_threshold ) or (G.nodes[v].get('degree',0) <= size_threshold ):
                continue

        w = float(a.get("weight", 0.0))
        if w >= min_weight:
            edges_kept.append((u, v, w))
    
    # 2) DETERMINA I NODI DA MANTENERE
    if hide_isolates:
        # Solo nodi che hanno almeno un arco
        nodes_kept = set()
        for u, v, _ in edges_kept:
            nodes_kept.add(u)
            nodes_kept.add(v)
    else:
        # Tutti i nodi del grafo
        nodes_kept = set(G.nodes())
    
    # 3) CREA IL SUBGRAFO
    G_sub = G.subgraph(nodes_kept).copy()
    
    # 4) CALCOLA RANGE PER SCALING
    if G_sub.number_of_nodes() > 0:
        deg_vals = [G_sub.nodes[n].get("degree", G_sub.degree(n)) for n in G_sub.nodes()]
        dmin, dmax = min(deg_vals), max(deg_vals)
    else:
        dmin, dmax = 0, 1
    
    if edges_kept:
        w_vals = [w for _, _, w in edges_kept]
        wmin, wmax = min(w_vals), max(w_vals)
    else:
        wmin, wmax = 0, 1
    
    # 5) CREA LA RETE PYVIS
    net = Network(
        height=height,
        width=width,
        directed=True,
        notebook=False,
        cdn_resources='remote',
        bgcolor='#ffffff',
        font_color='black'
    )
            
    # Configura la fisica direttamente nelle opzioni della rete
    if physics_enabled:
        net.barnes_hut(
            gravity=-3000,  # Ridotto per meno attrazione al centro
            central_gravity=0.1,  # Ridotto per permettere la separazione
            spring_length=250,  # Aumentato per più spazio
            spring_strength=0.01,  # Ridotto per meno rigidità
            damping=0.15,  # Aumentato per stabilizzare prima
            overlap=0.2  # Aumentato per evitare sovrapposizioni
        )
    
    # 7) AGGIUNGI I NODI CON POSIZIONI
    for n in G_sub.nodes():
        a = G_sub.nodes[n]
        degree = a.get("degree", G_sub.degree(n))
        
        # Scala la dimensione del nodo in base al grado
        size = scale(float(degree), float(dmin), float(dmax), 15, 50)
        
        color = a.get("color","black")
        source_model = a.get("model","UNKNOWN")
        position = a.get("position",-1)
        
        # Crea il titolo (tooltip) con informazioni dettagliate
        title = f"Label: {n}\nModel: {source_model}\nN.Articles: {degree}\nPosition:{position}"
        #title = f"<b>{n}</b><br>Modello: {source_model}<br>Grado: {degree}"
        
        # Ottieni posizione
        #pos = positions.get(n, {"x": 0, "y": 0})
        
        net.add_node(
            n,
            label=f"{icon_map[source_model]} {n}",
            value=size,  # Pyvis usa 'value' per la dimensione
            #borderWidth=5,
            #borderWidthSelected=10,
            color={
                "background": color,
                "border": "black"
                #"border": color_dict[source_model]
            },
            title=title,
            physics=physics_enabled  # Se physics=False, la posizione è fissa
        )
    
    # 8) AGGIUNGI GLI ARCHI
    for u, v, w in edges_kept:
        if u not in G_sub or v not in G_sub:
            continue
        
        # Scala lo spessore dell'arco in base al peso
        width = scale(float(w), float(wmin), float(wmax), 1.0, 8.0)
        
        # Tooltip per l'arco
        title = f"{u} ↔ {v}<br>Peso: {w:.3f}"
        
        color_edge = G_sub.nodes[u].get("color","black")

        net.add_edge(
            u, v,
            value=width,
            title=title,
            color=color_edge,
            shadow={
            "enabled": True,
            "color": "rgba(255,255,255,0.9)",  # halo chiaro
            "size": 6,
            "x": 0,
            "y": 0
            },
            weight=w,
            smooth=True,
            arrowStrikethrough=False
        )
    
    # 9) MOSTRA/NASCONDI PULSANTI DI CONTROLLO
    if show_buttons:
        try:
            net.show_buttons(filter_=['physics'])
        except:
            pass  # Ignora se non supportato
    
    if not physics_enabled:
        net.toggle_physics(False)
    
    # 10) SALVA IL FILE
    net.save_graph(filename)
    
    # 11) STATISTICHE
    stats = {
        "filename": filename,
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "displayed_nodes": G_sub.number_of_nodes(),
        "displayed_edges": len(edges_kept),
        "min_weight_threshold": min_weight,
        "hide_isolates": hide_isolates
    }
    
    print(f"\n{'='*60}")
    print(f"✓ Grafo esportato con successo!")
    print(f"{'='*60}")
    print(f"📁 File salvato: {filename}")
    print(f"📊 Statistiche:")
    print(f"   • Nodi totali nel grafo: {stats['total_nodes']}")
    print(f"   • Archi totali nel grafo: {stats['total_edges']}")
    print(f"   • Nodi visualizzati: {stats['displayed_nodes']}")
    print(f"   • Archi visualizzati: {stats['displayed_edges']}")
    print(f"   • Soglia peso minimo: {stats['min_weight_threshold']:.3f}")
    print(f"   • Nascondi isolati: {stats['hide_isolates']}")
    print(f"{'='*60}\n")
    
    return stats


def export_multiple_versions(G, base_filename="graph", thresholds=[0.0, 0.3, 0.5, 0.7]):
    """
    Esporta multiple versioni del grafo con diverse soglie di peso.
    
    Parametri:
    ----------
    G : networkx.Graph
        Il grafo da esportare
    base_filename : str
        Nome base per i file (default: "graph")
    thresholds : list
        Lista di soglie di peso da usare (default: [0.0, 0.3, 0.5, 0.7])
    
    Returns:
    --------
    list : Lista di statistiche per ogni versione esportata
    """
    all_stats = []
    
    for threshold in thresholds:
        filename = f"{base_filename}_threshold_{threshold:.2f}.html"
        stats = export_graph_to_html(
            G,
            filename=filename,
            min_weight=threshold,
            hide_isolates=True
        )
        all_stats.append(stats)
    
    print(f"\n✓ Esportate {len(thresholds)} versioni del grafo!")
    return all_stats

if __name__ == "__main__":
    
    export_graph_to_html(
        G_display,
        filename="esempio_grafo.html",
        min_weight=0.05,
        hide_isolates=True,
        show_buttons=False,
        filter_cluster=[2],
        size_threshold = 50,
    )


✓ Grafo esportato con successo!
📁 File salvato: esempio_grafo.html
📊 Statistiche:
   • Nodi totali nel grafo: 581
   • Archi totali nel grafo: 7600
   • Nodi visualizzati: 17
   • Archi visualizzati: 20
   • Soglia peso minimo: 0.050
   • Nascondi isolati: True



In [38]:
leiden_cluster['Cluster'].value_counts()

Cluster
1     25
2     22
0     20
3     19
8     18
5     12
4     10
6      8
20     8
9      7
7      7
10     5
11     5
18     5
23     4
22     4
14     3
12     3
13     3
25     3
28     3
39     3
21     3
16     3
30     3
17     3
19     3
15     3
24     2
26     2
27     2
29     2
32     2
31     2
35     2
33     2
34     2
36     2
37     2
38     2
40     2
41     2
42     2
43     2
44     2
Name: count, dtype: int64

### Considering the outlier

In [97]:
import pandas as pd
edges = pd.read_parquet('edges.parquet')
nodes = pd.read_parquet('nodes.parquet')

In [98]:
edges = edges[ ~ ( edges['Model 1 Label'].isin(leiden_cluster['Topic Label']) | edges['Model 2 Label'].isin(leiden_cluster['Topic Label'])) ]

In [99]:
enriched_edges = edges.merge(nodes,how='inner',left_on='Model 1 Label',right_on='Topic Label')
enriched_edges['N_Match'] = enriched_edges['Topic Closeness']  * enriched_edges['Degree']
enriched_edges = enriched_edges[ enriched_edges['N_Match'] > 5 ]

In [100]:
filtered_topics_1 = []
filtered_topics_2 = []
closeness = []

for _, edge in enriched_edges.iterrows():
    topic1 = edge['Model 1 Label']
    topic2 = edge['Model 2 Label']
    c = edge['Topic Closeness']
    bidirection = enriched_edges[
        (enriched_edges['Model 1 Label'] == topic2) &
        (enriched_edges['Model 2 Label'] == topic1) ]
    if not bidirection.empty:
        filtered_topics_1.append(topic1)
        filtered_topics_2.append(topic2)
        closeness.append(c)
        
filtered_edges = pd.DataFrame({'Model 1 Label': filtered_topics_1,
                    'Model 2 Label': filtered_topics_2,
                    'Topic Closeness':closeness})

In [101]:
filtered_edges

,Model 1 Label,Model 2 Label,Topic Closeness
0,Cov-2 Antibody Detection Test,Covid Testing Kits,0.411765
1,Saliva-Based Covid-19 Detection,Covid Testing Kits,0.292683
2,Postoperative Surgical Care in COVID-19 Patients,NHS Staffing Struggle in England,0.135135
3,Telehealth and Telemedicine Use at Home,NHS Staffing Struggle in England,0.112628
4,Influenza Burden and Impact,Scottish Pandemic Flu Response,0.046875
5,Influenza Burden and Impact,Australian Pandemic Response Strategies,0.024038
6,National Cancer Surveillance and Risk,NHS Staffing Struggle in England,0.014009
7,Scottish Pandemic Flu Response,Influenza Burden and Impact,0.125000
8,NHS Staffing Struggle in England,Postoperative Surgical Care in COVID-19 Patients,0.058275
9,NHS Staffing Struggle in England,Telehealth and Telemedicine Use at Home,0.053613


In [102]:
import networkx as nx
from pyvis.network import Network


def scale(v, vmin, vmax, out_min, out_max):
    """Scala un valore da un range a un altro."""
    if vmax == vmin:
        return (out_min + out_max) / 2
    return out_min + (v - vmin) * (out_max - out_min) / (vmax - vmin)


def export_graph_to_html(
    G, 
    filename="graph.html",
    min_weight=0.0,
    hide_isolates=True,
    height="800px",
    width="100%",
    physics_enabled=True,
    show_buttons=True,
    filter_cluster=None
):
    """
    Esporta un grafo NetworkX in un file HTML interattivo standalone.
    
    Parametri:
    ----------
    G : networkx.Graph
        Il grafo da esportare
    filename : str
        Nome del file HTML di output (default: "graph.html")
    min_weight : float
        Soglia minima per mostrare gli archi (default: 0.0)
    hide_isolates : bool
        Se True, nasconde i nodi senza archi dopo il filtro (default: True)
    height : str
        Altezza del grafo (default: "800px")
    width : str
        Larghezza del grafo (default: "100%")
    physics_enabled : bool
        Se True, abilita la simulazione fisica (default: True)
    show_buttons : bool
        Se True, mostra i controlli interattivi (default: True)
    
    Returns:
    --------
    dict : Statistiche del grafo esportato
    """

    if filter_cluster:
        filter_cluster.append(-1)

    # 1) FILTRA GLI ARCHI in base al peso minimo e al cluster
    edges_kept = []
    for u, v, a in G.edges(data=True):
        # Cluster isolation
        if filter_cluster:
            if ( G.nodes[u].get('cluster',-1) not in filter_cluster ) | ( G.nodes[v].get('cluster',-1) not in filter_cluster ) | (  G.nodes[u].get('cluster',-1) == -1 & G.nodes[v].get('cluster',-1) == -1  ) :
                continue

        w = float(a.get("weight", 0.0))
        if w >= min_weight:
            edges_kept.append((u, v, w))
    
    # 2) DETERMINA I NODI DA MANTENERE
    if hide_isolates:
        # Solo nodi che hanno almeno un arco
        nodes_kept = set()
        for u, v, _ in edges_kept:
            nodes_kept.add(u)
            nodes_kept.add(v)
    else:
        # Tutti i nodi del grafo
        nodes_kept = set(G.nodes())
    
    # 3) CREA IL SUBGRAFO
    G_sub = G.subgraph(nodes_kept).copy()
    
    # 4) CALCOLA RANGE PER SCALING
    if G_sub.number_of_nodes() > 0:
        deg_vals = [G_sub.nodes[n].get("degree", G_sub.degree(n)) for n in G_sub.nodes()]
        dmin, dmax = min(deg_vals), max(deg_vals)
    else:
        dmin, dmax = 0, 1
    
    if edges_kept:
        w_vals = [w for _, _, w in edges_kept]
        wmin, wmax = min(w_vals), max(w_vals)
    else:
        wmin, wmax = 0, 1
    
    # 5) CREA LA RETE PYVIS
    net = Network(
        height=height,
        width=width,
        directed=True,
        notebook=False,
        cdn_resources='remote',
        bgcolor='#ffffff',
        font_color='black'
    )
    
    # Configura la fisica direttamente nelle opzioni della rete
    if physics_enabled:
        net.barnes_hut(
            gravity=-3000,  # Ridotto per meno attrazione al centro
            central_gravity=0.1,  # Ridotto per permettere la separazione
            spring_length=250,  # Aumentato per più spazio
            spring_strength=0.01,  # Ridotto per meno rigidità
            damping=0.15,  # Aumentato per stabilizzare prima
            overlap=0.2  # Aumentato per evitare sovrapposizioni
        )
    
    # 7) AGGIUNGI I NODI CON POSIZIONI
    for n in G_sub.nodes():
        a = G_sub.nodes[n]
        degree = a.get("degree", G_sub.degree(n))
        
        # Scala la dimensione del nodo in base al grado
        size = scale(float(degree), float(dmin), float(dmax), 15, 50)
        
        color = a.get("color", "#999999")
        source_model = a.get("model","UNKNOWN")
        position = a.get("position",-1)
        
        # Crea il titolo (tooltip) con informazioni dettagliate
        title = f"Label: {n}\nModel: {source_model}\nN.Articles: {degree}\nPosition:{position}"
        #title = f"<b>{n}</b><br>Modello: {source_model}<br>Grado: {degree}"
        
        # Ottieni posizione
        #pos = positions.get(n, {"x": 0, "y": 0})
        
        net.add_node(
            n,
            label=str(n),
            value=size,  # Pyvis usa 'value' per la dimensione
            color=color,
            title=title,
            physics=physics_enabled  # Se physics=False, la posizione è fissa
        )
    
    # 8) AGGIUNGI GLI ARCHI
    for u, v, w in edges_kept:
        if u not in G_sub or v not in G_sub:
            continue
        
        # Scala lo spessore dell'arco in base al peso
        width = scale(float(w), float(wmin), float(wmax), 1.0, 8.0)
        
        # Tooltip per l'arco
        title = f"{u} ↔ {v}<br>Peso: {w:.3f}"
        
        net.add_edge(
            u, v,
            value=width,
            title=title,
            weight=w
        )
    
    # 9) MOSTRA/NASCONDI PULSANTI DI CONTROLLO
    if show_buttons:
        try:
            net.show_buttons(filter_=['physics'])
        except:
            pass  # Ignora se non supportato
    
    if not physics_enabled:
        net.toggle_physics(False)
    
    # 10) SALVA IL FILE
    net.save_graph(filename)
    
    # 11) STATISTICHE
    stats = {
        "filename": filename,
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "displayed_nodes": G_sub.number_of_nodes(),
        "displayed_edges": len(edges_kept),
        "min_weight_threshold": min_weight,
        "hide_isolates": hide_isolates
    }
    
    print(f"\n{'='*60}")
    print(f"✓ Grafo esportato con successo!")
    print(f"{'='*60}")
    print(f"📁 File salvato: {filename}")
    print(f"📊 Statistiche:")
    print(f"   • Nodi totali nel grafo: {stats['total_nodes']}")
    print(f"   • Archi totali nel grafo: {stats['total_edges']}")
    print(f"   • Nodi visualizzati: {stats['displayed_nodes']}")
    print(f"   • Archi visualizzati: {stats['displayed_edges']}")
    print(f"   • Soglia peso minimo: {stats['min_weight_threshold']:.3f}")
    print(f"   • Nascondi isolati: {stats['hide_isolates']}")
    print(f"{'='*60}\n")
    
    return stats


def export_multiple_versions(G, base_filename="graph", thresholds=[0.0, 0.3, 0.5, 0.7]):
    """
    Esporta multiple versioni del grafo con diverse soglie di peso.
    
    Parametri:
    ----------
    G : networkx.Graph
        Il grafo da esportare
    base_filename : str
        Nome base per i file (default: "graph")
    thresholds : list
        Lista di soglie di peso da usare (default: [0.0, 0.3, 0.5, 0.7])
    
    Returns:
    --------
    list : Lista di statistiche per ogni versione esportata
    """
    all_stats = []
    
    for threshold in thresholds:
        filename = f"{base_filename}_threshold_{threshold:.2f}.html"
        stats = export_graph_to_html(
            G,
            filename=filename,
            min_weight=threshold,
            hide_isolates=True
        )
        all_stats.append(stats)
    
    print(f"\n✓ Esportate {len(thresholds)} versioni del grafo!")
    return all_stats


# ESEMPIO D'USO (decommentare per testare)
if __name__ == "__main__":
    # Crea un grafo di esempio
    G = nx.DiGraph(description='Connected components') 

    for _, node in nodes.iterrows():
        G.add_node(
        node['Topic Label'],
        degree=node['Degree'],
        model=node['Model']
    )

    for _, edge in filtered_edges.iterrows():
        G.add_edge(
        edge['Model 1 Label'],
        edge['Model 2 Label'],
        weight=edge['Topic Closeness']
    )
    
    
    # Esporta una singola versione
    export_graph_to_html(
        G,
        filename="outlier.html",
        min_weight=0.0,
        hide_isolates=True,
        show_buttons=False,
        #filter_cluster=[9]
    )
    
    # Oppure esporta versioni multiple
    # export_multiple_versions(G, base_filename="esempio", thresholds=[0.0, 0.2, 0.5, 0.8])


✓ Grafo esportato con successo!
📁 File salvato: outlier.html
📊 Statistiche:
   • Nodi totali nel grafo: 581
   • Archi totali nel grafo: 14
   • Nodi visualizzati: 10
   • Archi visualizzati: 14
   • Soglia peso minimo: 0.000
   • Nascondi isolati: True

